## Prepočet chýbajúcich metrík

Súbor: prepocet_metrik.ipynb

Program: Hospodárska informatika

Vypracovala: Bc. Terézia Drengubiaková

Bakalárska práca: : Metódy strojového učenia pre včasnú predikciu geomagnetických búrok

Vedúci diplomovej práce: doc. Ing. Peter Butka, PhD.

Konzultanti: Ing. Viera Krešňáková, PhD., RNDr. Šimon Mackovjak, PhD.

In [2]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

# =========================================================
# Nastavenia gridu
# =========================================================
L_LIST = [6, 12, 18, 24, 30, 36, 42, 48]
H_LIST = [1, 2, 3, 4, 5, 6]
FOLD_IDS = [1, 2, 3, 4, 5]

OUT_DIR = "event_predictions_grid"
METRICS_PATH = "model_metrics_1.csv"
ACC_TOL_NT = 25.0

# 99% normálny interval
Z_99 = 2.576

# názov súboru: event_fold{fold}_L{L_idx}_H{H}.csv
PRED_FILE_RE = re.compile(r"^event_fold(\d+)_L(\d+)_H(\d+)\.csv$")


# =========================================================
# Pomocné funkcie
# =========================================================
def regression_acc_within_tol(y_true, y_pred, tol=25.0):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    if not np.any(m):
        return np.nan
    return float(np.mean(np.abs(y_true[m] - y_pred[m]) <= tol))


def get_existing_prediction_files():
    found = {}
    if not os.path.exists(OUT_DIR):
        raise FileNotFoundError(f"Adresár {OUT_DIR} neexistuje.")

    for fname in os.listdir(OUT_DIR):
        m = PRED_FILE_RE.match(fname)
        if not m:
            continue
        fold, L_idx, H = map(int, m.groups())
        found[(fold, L_idx, H)] = os.path.join(OUT_DIR, fname)
    return found


def add_99_interval_columns(tp: pd.DataFrame, z=Z_99):
    tp = tp.copy()

    required_cols = ["DST_pred", "sigma_nT"]
    missing = [c for c in required_cols if c not in tp.columns]
    if missing:
        raise ValueError(f"CSV nemá požadované stĺpce: {missing}")

    tp["DST_p005"] = tp["DST_pred"] - z * tp["sigma_nT"]
    tp["DST_p995"] = tp["DST_pred"] + z * tp["sigma_nT"]
    return tp


def compute_interval_metrics(y_true, lo, hi):
    y_true = np.asarray(y_true, dtype=float)
    lo = np.asarray(lo, dtype=float)
    hi = np.asarray(hi, dtype=float)

    m = np.isfinite(y_true) & np.isfinite(lo) & np.isfinite(hi)
    if not np.any(m):
        return np.nan, np.nan

    y = y_true[m]
    lo = lo[m]
    hi = hi[m]

    picp = float(np.mean((y >= lo) & (y <= hi)))
    mpiw = float(np.mean(hi - lo))
    return picp, mpiw


def build_metric_row_from_csv(csv_path, fold, L_idx, L_value, H, acc_tol_nt=25.0):
    tp = pd.read_csv(csv_path)

    required_cols = ["DST_true", "DST_pred"]
    missing = [c for c in required_cols if c not in tp.columns]
    if missing:
        raise ValueError(f"{os.path.basename(csv_path)} nemá stĺpce: {missing}")

    y_true = tp["DST_true"].to_numpy(dtype=float)
    y_pred = tp["DST_pred"].to_numpy(dtype=float)

    row = {
        "fold": int(fold),
        "L_idx": int(L_idx),
        "L_value": int(L_value),
        "H": int(H),
        "MSE_nT2": float(mean_squared_error(y_true, y_pred)),
        "RMSE_nT": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE_nT": float(mean_absolute_error(y_true, y_pred)),
        f"ACC_within_{int(acc_tol_nt)}nT": regression_acc_within_tol(y_true, y_pred, tol=acc_tol_nt),
        "N_test": int(np.isfinite(y_true).sum()),
    }

    # 90%
    if {"DST_p10", "DST_p90"}.issubset(tp.columns):
        picp90, mpiw90 = compute_interval_metrics(
            tp["DST_true"], tp["DST_p10"], tp["DST_p90"]
        )
        row["PICP_90"] = picp90
        row["MPIW_90_nT"] = mpiw90
    else:
        row["PICP_90"] = np.nan
        row["MPIW_90_nT"] = np.nan

    # 95%
    if {"DST_p025", "DST_p975"}.issubset(tp.columns):
        picp95, mpiw95 = compute_interval_metrics(
            tp["DST_true"], tp["DST_p025"], tp["DST_p975"]
        )
        row["PICP_95"] = picp95
        row["MPIW_95_nT"] = mpiw95
    else:
        row["PICP_95"] = np.nan
        row["MPIW_95_nT"] = np.nan

    # 99%
    if {"DST_p005", "DST_p995"}.issubset(tp.columns):
        picp99, mpiw99 = compute_interval_metrics(
            tp["DST_true"], tp["DST_p005"], tp["DST_p995"]
        )
        row["PICP_99"] = picp99
        row["MPIW_99_nT"] = mpiw99
    else:
        row["PICP_99"] = np.nan
        row["MPIW_99_nT"] = np.nan

    return row


def load_existing_metrics():
    if not os.path.exists(METRICS_PATH):
        return pd.DataFrame()

    df = pd.read_csv(METRICS_PATH)
    if df.empty:
        return df

    for c in ["fold", "L_idx", "H"]:
        if c in df.columns:
            df[c] = df[c].astype(int)

    return df


def upsert_metrics_row(metrics_df: pd.DataFrame, row: dict):
    row_df = pd.DataFrame([row])

    if metrics_df.empty:
        return row_df.copy()

    key_mask = (
        (metrics_df["fold"] == int(row["fold"])) &
        (metrics_df["L_idx"] == int(row["L_idx"])) &
        (metrics_df["H"] == int(row["H"]))
    )

    if key_mask.any():
        # prepíš existujúci riadok
        for col in row_df.columns:
            metrics_df.loc[key_mask, col] = row[col]
    else:
        metrics_df = pd.concat([metrics_df, row_df], ignore_index=True)

    return metrics_df


# =========================================================
# Hlavný beh: prejdi všetkých 240 modelov
# =========================================================
existing_pred_files = get_existing_prediction_files()
metrics_df = load_existing_metrics()

updated_csv_count = 0
updated_metrics_count = 0
missing_csv_count = 0
errors = []

total_expected = len(FOLD_IDS) * len(L_LIST) * len(H_LIST)
print(f"Očakávaný počet modelov: {total_expected}")
print(f"Nájdené predikčné CSV: {len(existing_pred_files)}")

for fold in FOLD_IDS:
    for L_idx, L_value in enumerate(L_LIST, start=1):
        for H in H_LIST:
            key = (fold, L_idx, H)
            csv_path = existing_pred_files.get(key)

            if csv_path is None:
                missing_csv_count += 1
                print(f"[MISSING CSV] fold={fold}, L_idx={L_idx}, H={H}")
                continue

            try:
                # 1) načítaj CSV
                tp = pd.read_csv(csv_path)

                # 2) dopočítaj 99% interval
                tp = add_99_interval_columns(tp, z=Z_99)

                # 3) ulož späť CSV
                tp.to_csv(csv_path, index=False)
                updated_csv_count += 1

                # 4) dopočítaj metriky vrátane 99%
                row = build_metric_row_from_csv(
                    csv_path=csv_path,
                    fold=fold,
                    L_idx=L_idx,
                    L_value=L_value,
                    H=H,
                    acc_tol_nt=ACC_TOL_NT,
                )

                # 5) update model_metrics.csv
                metrics_df = upsert_metrics_row(metrics_df, row)
                updated_metrics_count += 1

                print(f"[OK] fold={fold}, L={L_value}, L_idx={L_idx}, H={H}")

            except Exception as e:
                errors.append((fold, L_idx, H, str(e)))
                print(f"[ERROR] fold={fold}, L_idx={L_idx}, H={H}: {e}")

# zotrieď a ulož metriky
if not metrics_df.empty:
    sort_cols = [c for c in ["fold", "L_idx", "H"] if c in metrics_df.columns]
    metrics_df = metrics_df.sort_values(sort_cols).reset_index(drop=True)
    metrics_df.to_csv(METRICS_PATH, index=False)

print("\n===================== SUMMARY =====================")
print(f"Spracované CSV s doplneným 99% intervalom: {updated_csv_count}")
print(f"Aktualizované riadky v model_metrics.csv: {updated_metrics_count}")
print(f"Chýbajúce CSV: {missing_csv_count}")
print(f"Počet chýb: {len(errors)}")

if errors:
    err_df = pd.DataFrame(errors, columns=["fold", "L_idx", "H", "error"])
    display(err_df)
else:
    print("Bez chýb.")

display(metrics_df.tail(20))

Očakávaný počet modelov: 240
Nájdené predikčné CSV: 240
[OK] fold=1, L=6, L_idx=1, H=1
[OK] fold=1, L=6, L_idx=1, H=2
[OK] fold=1, L=6, L_idx=1, H=3
[OK] fold=1, L=6, L_idx=1, H=4
[OK] fold=1, L=6, L_idx=1, H=5
[OK] fold=1, L=6, L_idx=1, H=6
[OK] fold=1, L=12, L_idx=2, H=1
[OK] fold=1, L=12, L_idx=2, H=2
[OK] fold=1, L=12, L_idx=2, H=3
[OK] fold=1, L=12, L_idx=2, H=4
[OK] fold=1, L=12, L_idx=2, H=5
[OK] fold=1, L=12, L_idx=2, H=6
[OK] fold=1, L=18, L_idx=3, H=1
[OK] fold=1, L=18, L_idx=3, H=2
[OK] fold=1, L=18, L_idx=3, H=3
[OK] fold=1, L=18, L_idx=3, H=4
[OK] fold=1, L=18, L_idx=3, H=5
[OK] fold=1, L=18, L_idx=3, H=6
[OK] fold=1, L=24, L_idx=4, H=1
[OK] fold=1, L=24, L_idx=4, H=2
[OK] fold=1, L=24, L_idx=4, H=3
[OK] fold=1, L=24, L_idx=4, H=4
[OK] fold=1, L=24, L_idx=4, H=5
[OK] fold=1, L=24, L_idx=4, H=6
[OK] fold=1, L=30, L_idx=5, H=1
[OK] fold=1, L=30, L_idx=5, H=2
[OK] fold=1, L=30, L_idx=5, H=3
[OK] fold=1, L=30, L_idx=5, H=4
[OK] fold=1, L=30, L_idx=5, H=5
[OK] fold=1, L=30, L_i

,fold,L_idx,L_value,H,MSE_nT2,MAE_nT,ACC_within_25nT,RMSE_nT,PICP_90,MPIW_90_nT,PICP_95,MPIW_95_nT,N_test,PICP_99,MPIW_99_nT
220,5,5,30,5,79.166775,3.040594,0.984340,8.897571,NaN,NaN,NaN,NaN,1341,0.987323,35.211005
221,5,5,30,6,100.618525,3.323178,0.984305,10.030879,NaN,NaN,NaN,NaN,1338,0.989537,40.129236
222,5,6,36,1,8.150797,1.213325,0.997753,2.854960,NaN,NaN,NaN,NaN,1335,0.996255,11.795103
223,5,6,36,2,25.617027,1.938955,0.990991,5.061327,NaN,NaN,NaN,NaN,1332,0.990991,19.797372
224,5,6,36,3,43.896529,2.372978,0.989466,6.625446,NaN,NaN,NaN,NaN,1329,0.993228,25.869429
225,5,6,36,4,60.844172,2.789786,0.987179,7.800267,NaN,NaN,NaN,NaN,1326,0.992459,30.453620
226,5,6,36,5,77.947176,3.064650,0.984127,8.828770,NaN,NaN,NaN,NaN,1323,0.987906,35.010074
227,5,6,36,6,101.010151,3.532979,0.979545,10.050381,NaN,NaN,NaN,NaN,1320,0.988636,39.592576
228,5,7,42,1,8.667589,1.236109,0.997722,2.944077,NaN,NaN,NaN,NaN,1317,0.994685,11.387699
229,5,7,42,2,25.972821,1.967705,0.991629,5.096354,NaN,NaN,NaN,NaN,1314,0.991629,20.130957


In [3]:
import pandas as pd
import numpy as np

# =========================================================
# Cesty k súborom
# =========================================================
TARGET_PATH = "model_metrics.csv"          # súbor, kde chceš doplniť NaN
SOURCE_PATH = "model_metrics (1).csv"      # súbor, z ktorého sa majú brať správne hodnoty
OUTPUT_PATH = "model_metrics_filled.csv"   # výstupný opravený súbor

# Ak pracuješ s iným poškodeným súborom, napr.:
# TARGET_PATH = "model_metrics (2).csv"

# =========================================================
# Načítanie
# =========================================================
target = pd.read_csv(TARGET_PATH)
source = pd.read_csv(SOURCE_PATH)

key_cols = ["fold", "L_idx", "H"]

for c in key_cols:
    target[c] = target[c].astype(int)
    source[c] = source[c].astype(int)

# =========================================================
# Zoradenie podľa kľúčov
# =========================================================
target = target.sort_values(key_cols).reset_index(drop=True)
source = source.sort_values(key_cols).reset_index(drop=True)

# =========================================================
# Kontrola zhody riadkov
# =========================================================
target_keys = target[key_cols].astype(str).agg("|".join, axis=1)
source_keys = source[key_cols].astype(str).agg("|".join, axis=1)

if not target_keys.equals(source_keys):
    raise ValueError("Riadky v TARGET a SOURCE sa nezhodujú podľa ['fold', 'L_idx', 'H'].")

# =========================================================
# Doplnenie len NaN hodnôt
# =========================================================
common_cols = [c for c in source.columns if c in target.columns and c not in key_cols]

fill_stats = []

for col in common_cols:
    na_mask = target[col].isna() & source[col].notna()
    filled_count = int(na_mask.sum())

    if filled_count > 0:
        target.loc[na_mask, col] = source.loc[na_mask, col]

    fill_stats.append({
        "column": col,
        "filled_nans": filled_count
    })

# =========================================================
# Uloženie
# =========================================================
target.to_csv(OUTPUT_PATH, index=False)

print(f"Hotovo. Opravený súbor uložený ako: {OUTPUT_PATH}")

fill_stats_df = pd.DataFrame(fill_stats).sort_values("filled_nans", ascending=False)
display(fill_stats_df)

# voliteľne: ukáž problematickú oblasť okolo riadku 134
display(target.iloc[128:140])

Hotovo. Opravený súbor uložený ako: model_metrics_filled.csv


,column,filled_nans
8,MPIW_95_nT,107
5,PICP_90,107
7,PICP_95,107
6,MPIW_90_nT,107
1,MSE_nT2,0
0,L_value,0
4,RMSE_nT,0
3,ACC_within_25nT,0
2,MAE_nT,0
9,N_test,0


,fold,L_idx,L_value,H,MSE_nT2,MAE_nT,ACC_within_25nT,RMSE_nT,PICP_90,MPIW_90_nT,PICP_95,MPIW_95_nT,N_test,PICP_99,MPIW_99_nT
128,3,6,36,3,157.624226,4.340650,0.975922,12.554849,0.971407,29.172154,0.977427,34.758311,1329,0.986456,53.201497
129,3,6,36,4,215.082545,5.117820,0.966817,14.665693,0.969080,36.537099,0.974359,43.533565,1326,0.987179,66.633007
130,3,6,36,5,260.239718,5.886681,0.959184,16.131947,0.972789,42.734837,0.975813,50.918104,1323,0.987906,77.935873
131,3,6,36,6,301.609008,6.249415,0.958333,17.366894,0.971212,48.328733,0.976515,57.583172,1320,0.989394,88.137508
132,3,7,42,1,24.009484,2.002852,0.993166,4.899947,0.972665,12.639401,0.976462,15.059711,1317,0.989370,23.050578
133,3,7,42,2,90.954546,3.374843,0.985540,9.537009,0.971081,21.390512,0.977169,25.486568,1314,0.989346,39.010053
134,3,7,42,3,163.076184,4.404647,0.975591,12.770129,0.972540,30.228199,0.977117,36.016577,1311,0.987796,55.127414
135,3,7,42,4,203.451408,5.064131,0.967890,14.263639,0.972477,36.601663,0.978593,43.610492,1308,0.988532,66.750753
136,3,7,42,5,247.762060,5.658286,0.964751,15.740459,0.973180,41.967108,0.979310,50.003362,1305,0.988506,76.535759
137,3,7,42,6,285.016177,6.272046,0.958525,16.882422,0.970814,46.854521,0.976959,55.826663,1302,0.988479,85.448974
